<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

## Company Contact & Social Media Extraction

## Project Overview
**Task**: Generate a clean list of unique companies from executive_final.csv to prepare for automated collection of Customer Service emails, Investor Relations emails, and official social media pages (Facebook, X/Twitter, BlueSky).

**Dataset**: executives_final.csv  

---

## What This Pipeline Does?
1. Load: Reads the full executive dataset containing repeated company rows.
2. Filter: Removes all columns unrelated to company identity.
3. Deduplicate: Keeps only one row per company, producing a clean unique list.

Prepare: Outputs a CSV ready for the next task (automated web search for company contact information).

---

## Libraries Used

- **Packages**: `pandas`, `matplotlib`, `re`

---


## Pipeline

### Step 1: Configure / Knowing the data
```
INPUT_FILE = "executive_final.csv"
FINAL_FILE = "executive_titles_summary.csv"
```
### Step 2: Normalization
```
- Standardize text format (`.title()`)
- Remove punctuation and digits
- Create `title_clean` column
```
### Step 3: Categorization
Map each cleaned title to a normalized executive category

### Step 4: Visualization
- Bar Chart: Executive titles  
- Pie Chart: Distribution by executive category
- Histogram: Distribution of the executive titles

## Outputs:
1. CVS File: executive_titles_category_list 
2. Bar chatr: bar_chart.png
3. Pie chart: category_pie.png
4. Histogram: histogram_frequencies.png

# SECTION 1:SET-UP

In [1]:
#!pip install duckduckgo_search
#!pip install ddgs

In [2]:
# Libraries
import re
import time
import requests
import pandas as pd
import tldextract
import os
from urllib.parse import urlparse
from ddgs import DDGS
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

#SET UP: # Email regex
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

# HTTP headers
HEADERS = {"User-Agent": "Mozilla/5.0"}

# Noise / non-official domains
NOISE_DOMAINS = [
    "wikipedia.org",
    "yahoo.com",
    "news.yahoo.com",
    "bloomberg.com",
    "reuters.com",
    "marketwatch.com",
    "techcrunch.com",
    "crunchbase.com",
    "linkedin.com",
    "dwinnex.com",
    "wordpress.com",
    "blogspot.com",
    "medium.com",
    "zoominfo.com",
    "herokuapp.com",
    "sourceforge.net",
    "sec.gov",
    "bsky.app",       
    "facebook.com",
    "twitter.com",
    "x.com",
    "duckduckgo.com/y.js",
    "bing.com/aclick",
    "ad_domain=",
    "ad_provider=",
    "utm_",
    "doubleclick.net"
]

# Cache
ddg_cache = {}
domain_cache = {}
html_cache = {}

# SECTION 2: LOAD INPUT FILE

In [3]:
#Print some values:
# Configuration
INPUT_FILE = "executive_final.csv"
OUTPUT_FILE ="unique_companies.csv"
df = pd.read_csv(INPUT_FILE)
# Display preview
df_sample = df.head()
df_sample

,company,filing_type,executive_name,executive_title,confidence,executive_title_clean
0,1 800 FLOWERS COM INC,8-K,Moose Munch,Coo,spacy,Chief Operating Officer
1,"10x Genomics, Inc.",8-K,James Wilbur,Director,spacy,Director
2,1606 CORP.,8-K,Gregory Lambrecht,Chief Executive Officer,low,Chief Executive Officer
3,"1895 Bancorp of Wisconsin, Inc. /MD/",8-K,David Ball,Chief Executive Officer,spacy,Chief Executive Officer
4,"1stdibs.com, Inc.",8-K,Everette Taylor,Director,spacy,Director


In [4]:
companies = df[['company']].drop_duplicates()
companies = companies.sort_values(by='company').reset_index(drop=True)
companies.to_csv(OUTPUT_FILE, index=False)
companies.head()

,company
0,1 800 FLOWERS COM INC
1,"10x Genomics, Inc."
2,1606 CORP.
3,"1895 Bancorp of Wisconsin, Inc. /MD/"
4,"1stdibs.com, Inc."


# SECTION 3: Normalization of companies

In [5]:
# Normalization of companies

# Terms that are not acronyms
LEGAL_SUFFIXES = {
    "INC", "CORP", "CO", "LTD", "LLC", "PLC",
    "GROUP", "HOLDINGS", "HLDGS", "COM", "COMPANY"
}

# Simple domain pattern:
DOMAIN_RE = re.compile(
    r"^[A-Za-z0-9\-]+\.(com|org|net|io|co|ai|gov|edu|biz|info)$",
    flags=re.IGNORECASE
)

def is_acronym(word: str) -> bool:
    """
    True acronym:
    - 2–4 uppercase letters
    - letters only
    - not a legal suffix
    """
    return (
        word.isupper()
        and word.isalpha()
        and 2 <= len(word) <= 4
        and word not in LEGAL_SUFFIXES
    )

def is_mix_acronym(word: str) -> bool:
    """
    Mixed alphanumeric acronym like 3D, 5G, 2U.
    """
    return bool(re.match(r"^\d[A-Z]$", word))

def process_simple(part: str) -> str:
    """
    Normalize a single non-hyphen, non-domain token.
    """
    if not part:
        return part

    # Patterns like 8X8 to 8x8
    if re.match(r"^\d+[A-Za-z]\d+$", part):
        return part.lower()

    # True acronym (ETF, AAON, AB, FCP)
    if is_acronym(part):
        return part  # keep uppercase

    # Mixed acronym like 3D
    if is_mix_acronym(part):
        return part

    upper = part.upper()

    # Legal suffixes (Inc, Corp, Co, Ltd, LLC, PLC, etc.)
    if upper in LEGAL_SUFFIXES:
        if upper in {"LLC", "PLC"}:
            return upper  # keep uppercase for these
        else:
            return upper.capitalize() 

    # All caps but not acronym or legal suffix
    if part.isupper():
        return part.title()

    # Leading digits then letters
    if re.match(r"^\d+[A-Za-z]+$", part):
        digits = re.findall(r"^\d+", part)[0]
        letters = part[len(digits):]
        return digits + letters.capitalize()
    return part.capitalize()


def clean_token(word: str) -> str:
    """
    Token-level cleaning:
    - domain detection
    - dot cleanup for initials/suffixes
    - hyphen handling
    """
    if not word:
        return word

    # Domains
    if DOMAIN_RE.match(word):
        return word.lower()

    w = word

    # Remove dots in initials
    w = re.sub(r"\.(?=[A-Z]|$)", "", w)

    # Hyphenated words
    if "-" in w:
        parts = w.split("-")
        processed = [process_simple(p) for p in parts if p]
        return "-".join(processed)

    # Non-hyphen simple token
    return process_simple(w)


def normalize_company(name) -> str:
    """
    Full-company normalization:
    - remove SEC state codes (/DE/, /MD/, etc.)
    - remove commas
    - strip unwanted characters
    - collapse whitespace
    - normalize each token
    """
    if not isinstance(name, str):
        name = str(name)

    # Remove SEC state codes like
    name = re.sub(r"/[A-Za-z]{2}/", " ", name)

    # Remove commas
    name = name.replace(",", " ")

    # Remove unwanted characters except letters, digits, space, dot, ampersand, hyphen, slash
    name = re.sub(r"[^A-Za-z0-9\s.&\-/]", " ", name)

    # Collapse multiple spaces
    name = re.sub(r"\s+", " ", name).strip()

    if not name:
        return name

    tokens = name.split()
    cleaned_tokens = [clean_token(tok) for tok in tokens]

    return " ".join(cleaned_tokens)


# Apply to the deduplicated 'companies' dataframe
companies["company_clean"] = companies["company"].apply(normalize_company)

companies.head(30)

,company,company_clean
0,1 800 FLOWERS COM INC,1 800 Flowers Com Inc
1,"10x Genomics, Inc.",10X Genomics Inc
2,1606 CORP.,1606 Corp
3,"1895 Bancorp of Wisconsin, Inc. /MD/",1895 Bancorp Of Wisconsin Inc
4,"1stdibs.com, Inc.",1stdibs.com Inc
5,21Shares Core Ethereum ETF,21Shares Core Ethereum ETF
6,"22nd Century Group, Inc.",22Nd Century Group Inc
7,"2seventy bio, Inc.",2Seventy Bio Inc
8,374Water Inc.,374Water Inc
9,3D SYSTEMS CORP,3D Systems Corp


# SECTION 4: DOMAIN DISCOVERY HELPERS

In [6]:
def ddg_urls(query, max_results=8):
    """
    Cached DuckDuckGo wrapper that returns a list of URLs.
    """
    urls = []
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=max_results)
            for r in results:
                url = r.get("href", "")
                if not url:
                    continue

                # HARD FILTER: skip obvious ad/redirect scripts
                if "duckduckgo.com/y.js" in url:
                    continue
                if "aclick?" in url:
                    continue

                urls.append(url)
    except Exception:
        pass

    return urls


def is_noise(url):
    url = url.lower()

    # Block known noise domains
    for nd in NOISE_DOMAINS:
        if nd in url:
            return True

    # Block ad/tracking redirects
    ad_trash = [
        "duckduckgo.com/y.js",
        "bing.com/aclick",
        "ad_domain=",
        "ad_provider=",
        "utm_",
        "doubleclick.net",
        "clickserve",
        "tracking",
        "redirect"
    ]

    return any(pattern in url for pattern in ad_trash)


def extract_domain(url):
    if not url:
        return None
    try:
        ext = tldextract.extract(url)
        if not ext.domain or not ext.suffix:
            return None
        return f"{ext.domain}.{ext.suffix}".lower()
    except Exception:
        return None


def looks_like_company_domain(domain, company):
    """
    Score how much the domain resembles the company name.
    I use simple token overlap; more overlap = better.
    """
    if not domain or not company:
        return 0

    company_words = [
        w.lower()
        for w in re.split(r"\W+", company)
        if len(w) > 2 and not w.isdigit()
    ]
    if not company_words:
        return 0

    score = 0
    for w in company_words:
        if w in domain:
            score += 1
    return score


def discover_domain(company):
    """
    Domain discovery:
    - Rejects news/phishing/press-release domains
    - Strong token matching to ensure the domain resembles the company
    - Checks homepage availability before accepting a domain
    """

    if company in domain_cache:
        return domain_cache[company]

    # Remove punctuation, lowercase for comparisons
    clean_tokens = [
        t for t in re.split(r"[^a-z0-9]+", company.lower())
        if len(t) > 2 and not t.isdigit()
    ]

    if not clean_tokens:
        domain_cache[company] = None
        return None

    query_variants = [
        f"{company} official website",
        f"{company} corporate website",
        f"{company} homepage",
        f"{company} company website"
    ]

    candidate_domains = {}

    # Collect URL candidates
    for q in query_variants:
        urls = ddg_urls(q, max_results=20)

        for u in urls:
            if is_noise(u):
                continue

            dom = extract_domain(u)
            if not dom:
                continue

            # Score domain similarity
            score = 0
            dom_l = dom.lower()

            for token in clean_tokens:
                if token in dom_l:
                    score += 2  

            # Prefer the ".com"
            if dom.endswith(".com"):
                score += 1

            # Penalize weird extensions (.info, .co, .site, .store)
            if any(dom.endswith(ext) for ext in ["info", "site", "store", "online"]):
                score -= 3

            # Penalize too-short domains (1–2 letters)
            if len(dom.split(".")[0]) <= 2:
                score -= 4

            # Save
            if score > 0:
                candidate_domains[dom] = score

    if not candidate_domains:
        domain_cache[company] = None
        return None

    # Pick highest score
    best_domain = max(candidate_domains, key=candidate_domains.get)

    # Final validation: check homepage actually loads
    test_url = f"https://{best_domain}"
    html = fetch_html(test_url)

    if html:
        domain_cache[company] = best_domain
        return best_domain

    # Fallback: try http
    test_url = f"http://{best_domain}"
    html = fetch_html(test_url)
    if html:
        domain_cache[company] = best_domain
        return best_domain

    # If both fail just reject
    domain_cache[company] = None
    return None



# SECTION 5: HTLM Fetching / Email Helpers

In [7]:
def fetch_html(url):
    """
    Fetch HTML with basic retry and caching.
    """
    if not url:
        return None

    if url in html_cache:
        return html_cache[url]

    for attempt in range(2):
        try:
            r = requests.get(url, headers=HEADERS, timeout=8)
            if r.status_code == 200:
                html_cache[url] = r.text
                return r.text
        except Exception:
            time.sleep(1)

    html_cache[url] = None
    return None


def extract_emails_from_html(html):
    if not html:
        return []
    emails = re.findall(EMAIL_RE, html)
    return list(set(emails))
    
def find_page_on_domain(company, domain, keyword):
    """
    Use DuckDuckGo to find a page on the given domain for a keyword
    (e.g., 'investor relations', 'contact', 'support').
    """
    if not domain:
        return None

    query = f"{company} {keyword} site:{domain}"
    urls = ddg_urls(query, max_results=10)

    for u in urls:
        if domain in u.lower() and not is_noise(u):
            return u
    return None

# SECTION 6: IR AND CS EXTRACTORS

In [8]:
#EXTRACTORS:

#EMAIL:
def get_investor_info(company, domain):
    """
    Try to find an IR page and any email on it.
    Fallback to generic 'investor' or 'ir' keywords.
    """
    if not domain:
        return None, None

    # primary try
    page = find_page_on_domain(company, domain, "investor relations")
    if not page:
        # fallback keywords
        for kw in ["investor", "ir"]:
            page = find_page_on_domain(company, domain, kw)
            if page:
                break

    if not page:
        return None, None

    html = fetch_html(page)
    emails = extract_emails_from_html(html)

    # prefer emails that look like IR-related
    if emails:
        for e in emails:
            if any(tag in e.lower() for tag in ["ir@", "investor@", "investors@", "shareholder"]):
                return e, page
        return emails[0], page

    return None, page

#Customer Service:
def get_customer_service_info(company, domain):
    """
    Try to find a customer service / support / contact email.
    """
    if not domain:
        return None, None

    # primary keyword
    page = find_page_on_domain(company, domain, "customer service")
    if not page:
        # fallback keywords
        for kw in ["contact", "support", "help"]:
            page = find_page_on_domain(company, domain, kw)
            if page:
                break

    if not page:
        return None, None

    html = fetch_html(page)
    emails = extract_emails_from_html(html)

    # prefer 'support', 'service', 'help', 'info'
    if emails:
        for e in emails:
            if any(tag in e.lower() for tag in ["support@", "service@", "help@", "info@", "care@"]):
                return e, page
        return emails[0], page

    return None, page

# SECTION 7: EXTRACTOR LOOP

In [9]:
#EXTRACTOR LOOP:
def extract_for_company(row):
    """
    Given a row from companies, return a dict with extracted info.
    """
    company_raw = row["company"]
    company_clean = row["company_clean"]

    name_for_search = company_clean

    # Discover domain
    domain = discover_domain(name_for_search)

    ir_email, ir_page = None, None
    cs_email, cs_page = None, None

    if domain:
        ir_email, ir_page = get_investor_info(name_for_search, domain)
        cs_email, cs_page = get_customer_service_info(name_for_search, domain)

    return {
        "company": company_raw,
        "company_clean": company_clean,
        "domain": domain,
        "ir_page": ir_page,
        "ir_email": ir_email,
        "cs_page": cs_page,
        "cs_email": cs_email
    }

def run_production_extraction(companies,
                              out_file="company_contacts_full.csv",
                              max_companies=None):
    """
    Production-safe autosave version:
    - Writes EACH ROW immediately to CSV
    - Never loses progress
    """

    # If file exists, resume from it
    if os.path.exists(out_file):
        prev = pd.read_csv(out_file)
        completed = len(prev)
        print(f"Resuming from checkpoint at row {completed}")
    else:
        # Create empty CSV with header
        header_df = pd.DataFrame(columns=[
            "company", "company_clean", "domain",
            "ir_page", "ir_email",
            "cs_page", "cs_email",
            "error"
        ])
        header_df.to_csv(out_file, index=False)
        completed = 0
        print("Starting fresh extraction")

    # Select subset if needed
    if max_companies is not None:
        df_iter = companies.iloc[:max_companies].reset_index(drop=True)
    else:
        df_iter = companies.reset_index(drop=True)

    total = len(df_iter)

    # Extract row by row, appending immediately to CSV
    for idx in tqdm(range(completed, total), desc="Extracting"):
        row = df_iter.iloc[idx]

        try:
            info = extract_for_company(row)
            info["error"] = None
        except Exception as e:
            info = {
                "company": row["company"],
                "company_clean": row["company_clean"],
                "domain": None,
                "ir_page": None,
                "ir_email": None,
                "cs_page": None,
                "cs_email": None,
                "error": str(e)
            }

        # Convert to DataFrame for append
        pd.DataFrame([info]).to_csv(out_file, mode="a", header=False, index=False)

    print("Extraction completed.")
    print(f"Final results saved to: {out_file}")

    return pd.read_csv(out_file)

# RUN FULL EXTRACTOR

In [10]:
max_companies = None
output_file = "company_contacts_full.csv"

full_df = run_production_extraction(
    companies,
    out_file=output_file,
    max_companies=max_companies
)

full_df.head()


Resuming from checkpoint at row 4631


Extracting: 0it [00:00, ?it/s]

Extraction completed.
Final results saved to: company_contacts_full.csv


,company,company_clean,domain,ir_page,ir_email,cs_page,cs_email,error,twitter_url,facebook_url,bluesky_url
0,1 800 FLOWERS COM INC,1 800 Flowers Com Inc,1800flowersinc.com,https://www.1800flowersinc.com/investors,NaN,https://www.1800flowersinc.com/,NaN,NaN,NaN,https://www.facebook.com/1800flowers4566/,NaN
1,"10x Genomics, Inc.",10X Genomics Inc,10xgenomics.com,https://investors.10xgenomics.com/overview/def...,investors@10xgenomics.com,https://www.10xgenomics.com/support,support@10xgenomics.com,NaN,NaN,https://www.facebook.com/10xGenomics/,https://bsky.app/profile/did:plc:tdn46iumucy4s...
2,1606 CORP.,1606 Corp,1606restaurant.com,https://1606restaurant.com/reserve-a-table/,1606@beauporthotel.com,https://1606restaurant.com/reserve-a-table/,1606@beauporthotel.com,NaN,NaN,https://www.facebook.com/RedChipCompanies/post...,NaN
3,"1895 Bancorp of Wisconsin, Inc. /MD/",1895 Bancorp Of Wisconsin Inc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.facebook.com/OTCMarkets/videos/we-...,NaN
4,"1stdibs.com, Inc.",1stdibs.com Inc,1stdibs.com,https://investors.1stdibs.com/news/news-detail...,NaN,https://support.1stdibs.com/hc/en-us/articles/...,NaN,NaN,https://mobile.twitter.com/1stdibs,NaN,NaN


# SECTION 9: SOCIAL MEDIA EXTRACTOR

In [11]:
def find_twitter(company):
    urls = ddg_urls(f"{company} official twitter x", max_results=8)
    for u in urls:
        ul = u.lower()
        # avoid garbage links
        if ("twitter.com/" in ul or "x.com/" in ul) and \
           not any(bad in ul for bad in ["/status/", "/intent/", "/share", "/hashtag", "/search"]):
            return u
    return None


def find_facebook(company):
    urls = ddg_urls(f"{company} official facebook page", max_results=8)
    for u in urls:
        ul = u.lower()
        # avoid sharer.php or ad/tracking pages
        if "facebook.com/" in ul and \
           not any(bad in ul for bad in ["sharer", "php", "story.php", "l.php"]):
            return u
    return None


def find_bluesky(company):
    urls = ddg_urls(f"{company} official bluesky", max_results=8)
    for u in urls:
        if "bsky.app/profile/" in u.lower():
            return u
    return None

def run_social_extraction(
        in_file="company_contacts_full.csv",
        out_file="company_contacts_full.csv",
        max_companies=None):

    # Load full dataset
    df = pd.read_csv(in_file)

    # Ensure social media columns exist
    for col in ["twitter_url", "facebook_url", "bluesky_url"]:
        if col not in df.columns:
            df[col] = None

    # Determine resume point
    # A row is considered "complete" when ALL 3 social fields are non-null
    completed = df[["twitter_url", "facebook_url", "bluesky_url"]].notna().all(axis=1).sum()

    print(f"Resuming social extraction at row {completed}")

    if max_companies is None:
        total = len(df)
    else:
        total = min(max_companies, len(df))

    # Main loop
    for idx in tqdm(range(completed, total), desc="Extracting Social Media"):
        row = df.iloc[idx]
        company = row["company_clean"]

        # Perform extraction
        tw = find_twitter(company)
        fb = find_facebook(company)
        bs = find_bluesky(company)

        # Store results
        df.at[idx, "twitter_url"] = tw
        df.at[idx, "facebook_url"] = fb
        df.at[idx, "bluesky_url"] = bs

        # Save progress
        df.iloc[:idx+1].to_csv(out_file, index=False)

    print("Social media extraction completed.")
    return df


# SECTION 10: RUN SOCIAL EXTRACTOR

In [12]:
#social_df = run_social_extraction(
#    in_file="company_contacts_full.csv",
#    out_file="company_contacts_full.csv",
#    max_companies=None
#)
#social_df.head()

# SECTION 11: LOOK FOR MISSING DATA

In [13]:
# Load full extracted dataset
full = pd.read_csv("company_contacts_full.csv")

# Identify companies missing meaningful data
missing = full[
    (full["domain"].isna()) |
    (full["domain"].astype(str).str.len() < 3) |
    (full["ir_page"].isna()) |
    (full["cs_page"].isna())
].copy()

print(f"Total missing companies: {len(missing)}")

missing.head()

Total missing companies: 835


,company,company_clean,domain,ir_page,ir_email,cs_page,cs_email,error,twitter_url,facebook_url,bluesky_url
3,"1895 Bancorp of Wisconsin, Inc. /MD/",1895 Bancorp Of Wisconsin Inc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.facebook.com/OTCMarkets/videos/we-...,NaN
10,3M CO,3M Co,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.facebook.com/3M/,NaN
21,AA Mission Acquisition Corp.,AA Mission Acquisition Corp,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,ABERCROMBIE & FITCH CO /DE/,Abercrombie & Fitch Co,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.facebook.com/abercrombieofficial/,NaN
34,"ACELYRIN, Inc.",Acelyrin Inc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
def discover_domain(company):
    """
    Enterprise-grade corporate domain discovery.
    - strict brand-token filtering
    - multi-heuristic scoring
    - rejects platform/retail/support domains
    - fallback to homepage patterns
    - robust against AMD/Nvidia/multibrand contamination
    """

    if company in domain_cache:
        return domain_cache[company]

    clean = company.lower().strip()

    # Tokenize company name
    tokens = [t for t in re.split(r"\W+", clean) if len(t) > 2]
    if not tokens:
        domain_cache[company] = None
        return None

    # Generic terms to remove from scoring
    generic_terms = {"inc", "corp", "company", "group", "ltd", "co", "holdings", "holdings", "plc", "sa", "nv"}
    brand_tokens = [t for t in tokens if t not in generic_terms]

    if not brand_tokens:
        brand_tokens = tokens  # fallback: all tokens as brand tokens

    # Hard-reject platform-level domains
    BAD_BASE_DOMAINS = {
        "microsoft.com",
        "amazon.com",
        "google.com",
        "bing.com",
        "duckduckgo.com",
        "yahoo.com",
        "reddit.com",
        "facebook.com",
        "instagram.com",
        "walmart.com",
        "bestbuy.com",
        "newegg.com",
        "lenovo.com",
    }

    queries = [
        f"{company} official website",
        f"{company} corporate site",
        f"{company} homepage",
        f"{company} investor relations",
    ]

    best_domain = None
    best_score = 0

    for q in queries:
        urls = ddg_urls(q, max_results=10)

        for u in urls:
            if is_noise(u):
                continue

            dom = extract_domain(u)
            if not dom:
                continue

            # Hard reject platforms and marketplaces
            if dom in BAD_BASE_DOMAINS:
                continue

            # Score: presence of brand tokens in domain
            score = sum(1 for t in brand_tokens if t in dom)

            # Some brand tokens appear as initials (e.g., AMD, ADI, NVDA)
            initials = ''.join(t[0] for t in brand_tokens)
            if initials.lower() in dom:
                score += 1

            # Reject domains with zero brand-signal
            if score == 0:
                continue

            if score > best_score:
                best_score = score
                best_domain = dom

    # If we found a good match
    if best_domain:
        domain_cache[company] = best_domain
        return best_domain

    # ------------- Fallback Heuristics ----------------

    fallback = []

    # Try token0 + TLD patterns
    root = tokens[0]
    fallback += [
        f"{root}.com",
        f"{root}inc.com",
        f"{root}corp.com",
        f"{root}co.com",
        f"{root}group.com"
    ]

    # Try brand tokens (more reliable)
    for bt in brand_tokens:
        fallback.append(f"{bt}.com")

    # Search each fallback via DDG
    for fb in fallback:
        urls = ddg_urls(fb, max_results=3)
        for u in urls:
            dom = extract_domain(u)
            if dom == fb:
                domain_cache[company] = fb
                return fb

    domain_cache[company] = None
    return None


In [15]:
def extract_missing_info(row, idx):
    """
    Extracts ONLY missing fields from a row.
    Preserves any existing valid data.
    Returns a dict including the original row index (idx).
    """

    company_raw = row["company"]
    company_clean = row["company_clean"]

    # Start from existing values
    domain   = row.get("domain", None)
    ir_page  = row.get("ir_page", None)
    ir_email = row.get("ir_email", None)
    cs_page  = row.get("cs_page", None)
    cs_email = row.get("cs_email", None)
    twitter  = row.get("twitter_url", None)
    facebook = row.get("facebook_url", None)

    # 1. FIX DOMAIN
    if pd.isna(domain) or len(str(domain)) < 3:
        domain = discover_domain(company_clean)

    # 2. FIX IR PAGE & EMAIL
    if (pd.isna(ir_page) or str(ir_page).strip() == "") and domain:
        new_ir_email, new_ir_page = get_investor_info(company_clean, domain)
        if new_ir_page:
            ir_page = new_ir_page
        if new_ir_email:
            ir_email = new_ir_email

    # 3. FIX CUSTOMER SERVICE
    if (pd.isna(cs_page) or str(cs_page).strip() == "") and domain:
        new_cs_email, new_cs_page = get_customer_service_info(company_clean, domain)
        if new_cs_page:
            cs_page = new_cs_page
        if new_cs_email:
            cs_email = new_cs_email

    # 4. SOCIAL MEDIA (if you are still using this here)
    if ("twitter_url" in row) and (pd.isna(twitter) or twitter == ""):
        twitter = find_twitter(company_clean)

    if ("facebook_url" in row) and (pd.isna(facebook) or facebook == ""):
        facebook = find_facebook(company_clean)

    # Optional: debug print
    print("\n------------------------------------")
    print(f"idx={idx} | Company: {company_clean}")
    print(f"  Domain:                {domain}")
    print(f"  IR page:               {ir_page}")
    print(f"  IR email:              {ir_email}")
    print(f"  Customer Service page: {cs_page}")
    print(f"  Customer Service email:{cs_email}")
    print(f"  Twitter:               {twitter}")
    print(f"  Facebook:              {facebook}")
    print("------------------------------------\n")

    return {
        "idx": idx,
        "company": company_raw,
        "company_clean": company_clean,
        "domain": domain,
        "ir_page": ir_page,
        "ir_email": ir_email,
        "cs_page": cs_page,
        "cs_email": cs_email,
        "twitter_url": twitter,
        "facebook_url": facebook
    }


In [16]:
def run_missing_extraction(
    in_file="company_contacts_full.csv",
    out_file="company_contacts_full.csv",
    checkpoint_every=50
):
    # Load full dataset
    full = pd.read_csv(in_file)

    # Ensure social columns exist
    for col in ["twitter_url", "facebook_url"]:
        if col not in full.columns:
            full[col] = None

    # Define "missing" subset (companies needing fixes)
    missing = full[
        (full["domain"].isna()) |
        (full["domain"].astype(str).str.len() < 3) |
        (full["ir_page"].isna()) |
        (full["cs_page"].isna()) |
        (full["twitter_url"].isna()) |
        (full["facebook_url"].isna())
    ].copy()

    print(f"Total missing companies to fix: {len(missing)}")

    # Work on a copy we will update in-place
    full_fixed = full.copy()

    # Map from company name to row(s) in full_fixed
    # (assumes 'company' is unique; if not, all matching rows are updated)
    processed = 0

    for _, row in tqdm(missing.iterrows(), total=len(missing), desc="Fixing missing rows"):
        info = extract_missing_info(row)

        # Merge this row into full_fixed (by 'company')
        mask = (full_fixed["company"] == info["company"])
        for col in ["domain", "ir_page", "ir_email",
                    "cs_page", "cs_email",
                    "twitter_url", "facebook_url"]:
            full_fixed.loc[mask, col] = info[col]

        processed += 1

        # Checkpoint save
        if processed % checkpoint_every == 0:
            full_fixed.to_csv(out_file, index=False)
            print(f"Checkpoint saved after {processed} updates → {out_file}")

    # Final save
    full_fixed.to_csv(out_file, index=False)
    print(f"Done. Total updated rows: {processed}. Final saved to: {out_file}")

    return full_fixed


In [17]:
full_fixed = full.copy()

updated_batch = []
batch_size = 50

for i, (idx, row) in enumerate(tqdm(missing.iterrows(), total=len(missing), desc="Fixing missing rows")):
    # Extract new info for this row (idx is the index in full_fixed)
    info = extract_missing_info(row, idx)
    updated_batch.append(info)

    # Apply checkpoint every `batch_size` rows
    if (i + 1) % batch_size == 0:
        batch_df = pd.DataFrame(updated_batch)

        for _, r in batch_df.iterrows():
            ridx = r["idx"]
            full_fixed.loc[ridx, ["company",
                                  "company_clean",
                                  "domain",
                                  "ir_page",
                                  "ir_email",
                                  "cs_page",
                                  "cs_email",
                                  "twitter_url",
                                  "facebook_url"]] = [
                r["company"],
                r["company_clean"],
                r["domain"],
                r["ir_page"],
                r["ir_email"],
                r["cs_page"],
                r["cs_email"],
                r["twitter_url"],
                r["facebook_url"]
            ]

        full_fixed.to_csv("company_contacts_full.csv", index=False)
        print(f"Checkpoint saved after {i+1} missing rows.")
        updated_batch = []  # clear batch

# Apply remaining updates (if any) at the end
if updated_batch:
    batch_df = pd.DataFrame(updated_batch)

    for _, r in batch_df.iterrows():
        ridx = r["idx"]
        full_fixed.loc[ridx, ["company",
                              "company_clean",
                              "domain",
                              "ir_page",
                              "ir_email",
                              "cs_page",
                              "cs_email",
                              "twitter_url",
                              "facebook_url"]] = [
            r["company"],
            r["company_clean"],
            r["domain"],
            r["ir_page"],
            r["ir_email"],
            r["cs_page"],
            r["cs_email"],
            r["twitter_url"],
            r["facebook_url"]
        ]

    full_fixed.to_csv("company_contacts_full.csv", index=False)
    print("Final checkpoint saved (last partial batch).")

print("Missing-company extraction completed.")


Fixing missing rows:   0%|          | 0/835 [00:00<?, ?it/s]


------------------------------------
idx=3 | Company: 1895 Bancorp Of Wisconsin Inc
  Domain:                1895bancorpofwisconsin.com
  IR page:               https://www.1895bancorpofwisconsin.com/news-market-information/presentations/presentation-details/2024/Investor-Presentation/default.aspx
  IR email:              nan
  Customer Service page: https://www.1895bancorpofwisconsin.com/news-market-information/press-releases/news-details/2021/1895-Bancorp-of-Wisconsin-Inc.-Announces-Completion-of-Subscription-Offering/default.aspx
  Customer Service email:nan
  Twitter:               None
  Facebook:              https://www.facebook.com/OTCMarkets/videos/we-are-thrilled-to-welcome-1895-bancorp-of-wisconsin-inc-otcqx-bcow-the-holding-/3076816265807466/
------------------------------------


------------------------------------
idx=10 | Company: 3M Co
  Domain:                None
  IR page:               nan
  IR email:              nan
  Customer Service page: nan
  Customer Servic